<a href="https://colab.research.google.com/github/iOmarz16/career-copilot/blob/main/career_copilot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install -q -U "crewai[anthropic]" tavily-python gradio

import os
from getpass import getpass

def load_key(name):
    if os.environ.get(name): return
    value = None
    try:
        from google.colab import userdata
        value = userdata.get(name)
    except Exception:
        value = None
    if not isinstance(value, str) or not value.strip():
        value = getpass(f"Paste your {name}: ")
    os.environ[name] = value.strip()

load_key("ANTHROPIC_API_KEY")
load_key("TAVILY_API_KEY")
print("Keys loaded ✅")

In [ ]:
from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool
from tavily import TavilyClient
import gradio as gr

load_key("N8N_WEBHOOK_URL")  # add it in Colab Secrets
N8N_WEBHOOK_URL = os.environ["N8N_WEBHOOK_URL"]
llm = LLM(model="anthropic/claude-haiku-4-5", temperature=0.4, max_tokens=2048)

# The web-search tool (same as the main Day 4 notebook)
_tavily = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

@tool("Web Search")
def web_search(query: str) -> str:
    """Search the live web for current information. Input a short search query."""
    try:
        rs = _tavily.search(query=query, max_results=3)["results"]
        return "\n\n".join(f"TITLE: {r['title']}\nINFO: {r['content'][:300]}\nSOURCE: {r['url']}" for r in rs)
    except Exception as e:
        return f"Error: search failed ({e})."

print("Setup done 🛠️")

In [ ]:
researcher = Agent(
    role="Career Research Assistant",
    goal="Find accurate, current information about a company and job role by searching the web.",
    backstory=(
        "You research companies, job roles, and interview expectations thoroughly. "
        "You never invent facts and always keep source links."
    ),
    llm=llm,
    tools=[web_search],
    verbose=False,
)

writer = Agent(
    role="Career Coach Writer",
    goal="Turn research into a clear, well-organized interview prep report for a student.",
    backstory=(
        "You write clear, structured career reports with a short overview, "
        "key interview questions, skills to review, CV tips, and sources."
    ),
    llm=llm,
    verbose=False,
)

def build_crew(company, job_role, cv_text=""):
    research = Task(
        description=(
            f"Search the web for current, accurate information about the company '{company}' "
            f"and the job role '{job_role}'. Gather 4-5 key facts, including company overview, "
            "role responsibilities, required skills, and any relevant recent information. "
            "Include source links for each important fact."
        ),
        expected_output="4-5 key facts about the company and role, each with a source link.",
        agent=researcher,
    )

    report = Task(
        description=(
            f"Write a career prep report for '{company}' and '{job_role}' using the research. "
            "Structure it as: 1) a 2-sentence company/job intro, 2) 4-5 bullet-point key facts, "
            "3) 5 likely interview questions, 4) short model answers, "
            "5) CV improvement tips, and 6) a Sources list at the end. "
        ),
        expected_output=(
            "A structured career prep report with intro, key facts, questions, answers, tips, and sources."
        ),
        agent=writer,
        context=[research],
    )

    return Crew(
        agents=[researcher, writer],
        tasks=[research, report],
        process=Process.sequential
    )

def run_career_copilot(company, job_role, cv_text):
    crew = build_crew(company, job_role, cv_text)
    result = crew.kickoff()
    output = result.raw if hasattr(result, "raw") else str(result)

    payload = {
        "company": company,
        "job_role": job_role,
        "cv_text": cv_text,
        "career_report": output
    }

    try:
        requests.post(N8N_WEBHOOK_URL, json=payload, timeout=30)
    except Exception as e:
        output += f"\n\n[n8n send failed: {e}]"

    return output

with gr.Blocks() as demo:
    gr.Markdown("#  Career Copilot")

    with gr.Row():
        company_in = gr.Textbox(label="Company", placeholder="e.g., STC, Aramco, NEOM")
        role_in = gr.Textbox(label="Job Role", placeholder="e.g., Data Science Intern")



    run_btn = gr.Button("Generate Career Report", variant="primary")
    output = gr.Textbox(label="Output", lines=24)

    run_btn.click(
        fn=run_career_copilot,
        inputs=[company_in, role_in, cv_in],
        outputs=output
    )

demo.launch(share=True)